In [ ]:
import ast
from pathlib import Path

import pandas as pd

ANNOTATORS_CSV = Path("datasets/ukrainian/annotations/annotators.csv")

raw = pd.read_csv(ANNOTATORS_CSV)
raw["labels_annotator1"] = raw["labels_annotator1"].apply(ast.literal_eval)
raw["labels_annotator2"] = raw["labels_annotator2"].apply(ast.literal_eval)

all_labels = sorted({l for col in ("labels_annotator1", "labels_annotator2")
                     for labels in raw[col] for l in labels})

def to_binary(label_lists, columns):
    df = pd.DataFrame(0, index=range(len(label_lists)), columns=columns, dtype=int)
    for i, labels in enumerate(label_lists):
        for l in labels:
            if l in df.columns:
                df.at[i, l] = 1
    return df

df1 = to_binary(raw["labels_annotator1"], all_labels)
df2 = to_binary(raw["labels_annotator2"], all_labels)
df1.insert(0, "filename", raw["image"])
df2.insert(0, "filename", raw["image"])

assert df1.shape == df2.shape
print(f"{len(df1)} items, {len(all_labels)} labels")


In [ ]:
exact_matches = sum(
    1 for i in range(len(df1))
    if frozenset(df1.columns[1:][df1.iloc[i, 1:] == 1]) ==
       frozenset(df2.columns[1:][df2.iloc[i, 1:] == 1])
)
print(f"Exact matches: {exact_matches} / {len(df1)}")


## Bootstrapped multi-label IAA (Marchal et al. 2022)

Port of the R reference implementation: https://osf.io/f5v4p/

In [ ]:
import numpy as np
from nltk.metrics.agreement import AnnotationTask
from nltk.metrics import masi_distance
from sklearn.metrics import cohen_kappa_score


def safe_masi(label1, label2):
    if not label1 and not label2:
        return 0.0
    if not label1 or not label2:
        return 1.0
    return masi_distance(label1, label2)


def multilabel_agreement(df1, df2, n_bootstrap=500, seed=42):
    """Bootstrapped multi-label IAA. Precision = |∩| / |c2|, recall = |∩| / |c1| (per Marchal 2022)."""
    rng = np.random.default_rng(seed)
    a1 = df1.values.astype(int)
    a2 = df2.values.astype(int)
    n_items, n_classes = a1.shape

    def compute_measures(m1, m2):
        precisions, recalls, f1s, intersections = [], [], [], []
        for i in range(len(m1)):
            s1 = set(np.where(m1[i] == 1)[0])
            s2 = set(np.where(m2[i] == 1)[0])
            inter = len(s1 & s2)
            intersections.append(1 if inter > 0 else 0)
            p = inter / len(s2) if s2 else 0.0
            r = inter / len(s1) if s1 else 0.0
            f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
            precisions.append(p)
            recalls.append(r)
            f1s.append(f1)
        return {
            "intersection": np.mean(intersections),
            "precision":    np.mean(precisions),
            "recall":       np.mean(recalls),
            "f1":           np.mean(f1s),
        }

    observed = compute_measures(a1, a2)

    total1, total2 = a1.sum(), a2.sum()
    p1 = a1.sum(axis=0) / total1 if total1 > 0 else np.ones(n_classes) / n_classes
    p2 = a2.sum(axis=0) / total2 if total2 > 0 else np.ones(n_classes) / n_classes

    nlabels1 = a1.sum(axis=1)
    nlabels2 = a2.sum(axis=1)

    boot = {"intersection": [], "precision": [], "recall": [], "f1": []}
    for _ in range(n_bootstrap):
        sim1 = np.zeros((n_items, n_classes), dtype=int)
        sim2 = np.zeros((n_items, n_classes), dtype=int)
        for i in range(n_items):
            nl1 = min(int(rng.choice(nlabels1)), n_classes)
            nl2 = min(int(rng.choice(nlabels2)), n_classes)
            if nl1 > 0:
                sim1[i, rng.choice(n_classes, size=nl1, replace=False, p=p1)] = 1
            if nl2 > 0:
                sim2[i, rng.choice(n_classes, size=nl2, replace=False, p=p2)] = 1
        m = compute_measures(sim1, sim2)
        for k in boot:
            boot[k].append(m[k])

    expected = {k: np.mean(v) for k, v in boot.items()}

    def adj(o, e):
        return (o - e) / (1 - e) if e < 1 else (1.0 if o >= 1 else 0.0)

    adjusted = {k: adj(observed[k], expected[k]) for k in observed}

    # Augmented kappa: each label weighted by 1 / (n_labels for that item).
    obs_aug = 0.0
    for i in range(n_items):
        s1 = np.where(a1[i] == 1)[0]
        s2 = np.where(a2[i] == 1)[0]
        if len(s1) == 0 or len(s2) == 0:
            continue
        for k in set(s1) & set(s2):
            obs_aug += (1.0 / len(s1)) * (1.0 / len(s2))
    obs_aug /= n_items

    w1 = np.zeros(n_classes)
    w2 = np.zeros(n_classes)
    for i in range(n_items):
        n1, n2 = a1[i].sum(), a2[i].sum()
        if n1 > 0:
            w1 += a1[i] / n1
        if n2 > 0:
            w2 += a2[i] / n2
    w1 /= n_items
    w2 /= n_items
    exp_aug = (w1 * w2).sum()
    aug_kappa = adj(obs_aug, exp_aug)

    shared_cols = df1.columns.intersection(df2.columns)
    per_class_kappa = {col: cohen_kappa_score(df1[col], df2[col]) for col in shared_cols}

    masi_data = []
    for i in range(n_items):
        labels1 = frozenset(df1.columns[df1.iloc[i] == 1])
        labels2 = frozenset(df2.columns[df2.iloc[i] == 1])
        masi_data.append(("c1", str(i), labels1))
        masi_data.append(("c2", str(i), labels2))
    masi_alpha = AnnotationTask(data=masi_data, distance=safe_masi).alpha()

    return {
        "observed":         observed,
        "expected":         expected,
        "adjusted":         adjusted,
        "augmented_kappa":  {"observed": obs_aug, "expected": exp_aug, "adjusted": aug_kappa},
        "per_class_kappa":  per_class_kappa,
        "masi_alpha":       masi_alpha,
    }


def print_results(res):
    print(f"{'Measure':<18} {'Observed':>10} {'Expected':>10} {'Adjusted':>10}")
    print("-" * 50)
    name_map = {"intersection": "boot-match", "precision": "boot-prec",
                "recall": "boot-recall", "f1": "boot-F1"}
    for k in ["intersection", "precision", "recall", "f1"]:
        print(f"{name_map[k]:<18} {res['observed'][k]:>10.3f} "
              f"{res['expected'][k]:>10.3f} {res['adjusted'][k]:>10.3f}")

    ak = res["augmented_kappa"]
    print(f"{'augmented-κ':<18} {ak['observed']:>10.3f} {ak['expected']:>10.3f} {ak['adjusted']:>10.3f}")

    print(f"\nMASI Krippendorff α: {res['masi_alpha']:.3f}")

    kappas = res["per_class_kappa"]
    print(f"\nPer-class Cohen's κ (mean: {np.mean(list(kappas.values())):.3f}):")
    for col, k in sorted(kappas.items(), key=lambda x: -x[1]):
        print(f"  {col:<25} κ = {k:.3f}")


In [ ]:
res = multilabel_agreement(
    df1.drop(columns=["filename"]),
    df2.drop(columns=["filename"]),
    n_bootstrap=1000,
    seed=42,
)
print_results(res)
